# Larger systems & higher cardinality

Beyond binary $2\times 2$: ternary / categorical treatments, longer chains,
multi-cause systems, and multi-instrument IV. Uses the literature problem
catalog in `partialid.problems`.


In [ ]:
import time
from partialid import Assumptions, Query, Variables, bound, bound_path
from partialid.problems import (
    all_literature_problems,
    chain_order_only,
    shridharan_multi_treatment,
    ternary_cardinalities,
    ternary_exposure,
    two_instruments,
)

def show(label, r, Q=None, dt_ms=None):
    timing = f"  ({dt_ms:.1f} ms)" if dt_ms is not None else ""
    head = Q.describe() if Q is not None else ""
    print(f"{label}{timing}")
    if head:
        print(f"  query: {head}")
    print(f"  [{r.lower:.4f}, {r.upper:.4f}]  w={r.width:.4f}  method={r.method}  tight={r.tight}")


## Ternary confounded exposure

$X \in \{0,1,2\}$, binary $Y$. Under order-only, Manski width $= 1 - P(X=x^\star)$.


In [ ]:
p = ternary_exposure(seed=0)
t0 = time.perf_counter()
r = bound(p.variables, p.data, p.query, p.assumptions)
dt = (time.perf_counter() - t0) * 1000
show(p.name, r, p.query, dt)
print(f"  expected_width={p.expected_width:.4f}  |domains|={ {k: len(v) for k,v in p.variables.domains.items()} }")
assert abs(r.width - p.expected_width) < 1e-6


## Scaling treatment / outcome cardinality


In [ ]:
for card_x, card_y in [(4, 2), (5, 2), (3, 3), (4, 3)]:
    p = ternary_cardinalities(card_x, card_y, seed=1)
    t0 = time.perf_counter()
    r = bound(p.variables, p.data, p.query, p.assumptions)
    dt = (time.perf_counter() - t0) * 1000
    n_types_proxy = card_x ** 1 * card_y ** card_x  # rough response-type scale for Y|X
    print(
        f"X={card_x} Y={card_y}: w={r.width:.4f}  method={r.method:20s}  "
        f"{dt:6.1f} ms  cells={len(p.data)}"
    )


## MSM sensitivity on a 4-level treatment


In [ ]:
p = ternary_cardinalities(4, 2, seed=1)
a = Assumptions.order("X", "Y")
print("Manski:", bound(p.variables, p.data, p.query, a).width)
for lam in (1.0, 1.5, 2.0, 3.0):
    r = bound(p.variables, p.data, p.query, a & Assumptions.tan_msm("X", lam))
    print(f"  Λ={lam:g}: [{r.lower:.4f}, {r.upper:.4f}] w={r.width:.4f}  method={r.method}")


## Order-only chains (inert-variable pruning)

Longer chains under `Assumptions.order` stay fast because inert intermediates
are pruned before forming the type LP.


In [ ]:
for n, card in [(3, 2), (4, 2), (5, 2), (3, 3), (4, 3)]:
    p = chain_order_only(n, card, seed=7)
    t0 = time.perf_counter()
    r = bound(p.variables, p.data, p.query, p.assumptions)
    dt = (time.perf_counter() - t0) * 1000
    print(
        f"chain n={n} card={card}: w={r.width:.4f}  method={r.method:12s}  "
        f"{dt:6.1f} ms  vars={list(p.variables.names)}"
    )


## Multi-cause / multi-treatment


In [ ]:
for n_t, card in [(2, 2), (3, 2)]:
    p = shridharan_multi_treatment(n_t, card, seed=6)
    t0 = time.perf_counter()
    r = bound(p.variables, p.data, p.query, p.assumptions)
    dt = (time.perf_counter() - t0) * 1000
    show(p.name, r, p.query, dt)


## Two-instrument IV


In [ ]:
p = two_instruments(seed=3)
t0 = time.perf_counter()
r = bound(p.variables, p.data, p.query, p.assumptions)
dt = (time.perf_counter() - t0) * 1000
show(p.name, r, p.query, dt)
print(f"  |Z1×Z2×X×Y cells|={len(p.data)}")


## Literature catalog sweep

`all_literature_problems()` returns a mid-size battery used in benchmarks.


In [ ]:
rows = []
for p in all_literature_problems():
    t0 = time.perf_counter()
    r = bound(p.variables, p.data, p.query, p.assumptions)
    dt = (time.perf_counter() - t0) * 1000
    rows.append((p.name, r.width, r.method, r.tight, dt, len(p.data)))
print(f"{'name':32s} {'width':>8s} {'method':24s} {'tight':5s} {'ms':>8s} {'cells':>5s}")
for name, w, method, tight, dt, cells in rows:
    print(f"{name:32s} {w:8.4f} {method:24s} {str(tight):5s} {dt:8.1f} {cells:5d}")


## Sensitivity path on ternary exposure


In [ ]:
p = ternary_exposure(seed=0)
path = bound_path(
    p.variables, p.data, p.query, Assumptions.manski("X", "Y"),
    param="Lambda", theta_min=1.0, theta_max=4.0, n_grid=7,
)
print(path.summary(at=2.0))
for lam in (1.0, 2.0, 3.0, 4.0):
    lo, hi, w = path.at(lam)
    print(f"  Λ={lam:g}: [{lo:.4f}, {hi:.4f}] w={w:.4f}")
